# Ridge decoder grid & subset analysis

Neuron-subset decoding performance for the cut-treadmill Ridge decoder, across V1 motor
sessions of `ccyp_l5_3d_vision` and `colasa_3d-vision_revisions`.

Targets: Optic Flow (OF), Running Speed (RS), Depth, and the depth-orthogonal **RS×OF**
product (`rsof_product_stim`). Depth is the RS/OF ratio in log space; RS×OF is the axis
orthogonal to it.

Sections:
1. Subset performance across **all** conditions, colour-coded by recording depth.
2. Subset performance on the two balanced condition **grids** (Grid 1: high OF, Grid 2: low OF).

All results use the cut-treadmill subsets (`_motor_cut`).

In [ ]:
%load_ext autoreload
%autoreload 2

import warnings
warnings.filterwarnings("ignore")

import flexiznam as flz
import pandas as pd
import numpy as np
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import seaborn as sns
from pathlib import Path

from cottage_analysis.summary_analysis import load_project_subsets

# Style
sns.set_theme(style="ticks", context="talk")
font_path = '/nemo/lab/znamenskiyp/home/shared/resources/fonts/arial.ttf'
fm.fontManager.addfont(font_path)
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = 'Arial'

PROJECTS = ["ccyp_l5_3d_vision", "colasa_3d-vision_revisions"]
CUT_OFF = 350  # nominal depth (um) splitting Layer 2/3 from Layer 5

SESSIONS_TO_EXCLUDE = {
    "PZAG22.1b_S20260220": "1000 more frames than triggers in the treadmill recording, recording was wrongly started"
}

# All decoder targets, with display labels and consistent colours.
TARGETS = ["OF_stim", "RS_stim", "depth", "rsof_product_stim"]
TARGET_LABEL = {
    "OF_stim": "Optic Flow (OF)",
    "RS_stim": "Running Speed (RS)",
    "depth": "Depth",
    "rsof_product_stim": "RS×OF (depth-orthogonal)",
}
TARGET_COLOR = {"OF_stim": "C0", "RS_stim": "C1", "depth": "C2", "rsof_product_stim": "C3"}

## 1. Subset performance across all conditions, by recording depth

In [ ]:
# Load the all-conditions cut-treadmill subset results for both projects
dfs = []
for project in PROJECTS:
    df = load_project_subsets(project, session_to_exclude=SESSIONS_TO_EXCLUDE.keys(), cut_treadmill=True)
    df["project"] = project
    dfs.append(df)
subsets_all_df = pd.concat(dfs, ignore_index=True)

print("Targets:", subsets_all_df["target"].unique())
print("Conditions:", subsets_all_df["condition"].unique())
print("Sessions:", subsets_all_df["session"].nunique())

In [ ]:
def plot_subsets_vs_depth(metric_mean, metric_std, suptitle, ylabel, ylim=None):
    """One panel per target: per-session subset-size curve, coloured by recording depth."""
    depths = sorted(subsets_all_df["nominal_depth"].dropna().unique())
    norm = mcolors.Normalize(vmin=min(depths), vmax=max(depths))
    cmap = cm.viridis
    cond = "closedloop"

    fig, axes = plt.subplots(1, len(TARGETS), figsize=(5.2 * len(TARGETS), 5), sharey=True)
    for ax, target in zip(axes, TARGETS):
        cond_df = subsets_all_df[(subsets_all_df["target"] == target) & (subsets_all_df["condition"] == cond)]
        for _, sess_df in cond_df.groupby("session"):
            sess_df = sess_df.sort_values("subset_size")
            depth = sess_df["nominal_depth"].iloc[0]
            if pd.isna(depth):
                continue
            color = cmap(norm(depth))
            ax.plot(sess_df["subset_size"], sess_df[metric_mean], marker='o', color=color, alpha=0.75, linewidth=2)
            ax.fill_between(sess_df["subset_size"],
                            sess_df[metric_mean] - sess_df[metric_std],
                            sess_df[metric_mean] + sess_df[metric_std], color=color, alpha=0.08)
        ax.set_title(TARGET_LABEL[target]); ax.set_xlabel("Subset Size (Neurons)")
        ax.grid(True, linestyle="--", alpha=0.5)
        if ylim is not None:
            ax.set_ylim(*ylim)
    axes[0].set_ylabel(ylabel)
    sm = cm.ScalarMappable(cmap=cmap, norm=norm); sm.set_array([])
    fig.colorbar(sm, ax=axes, label='Recording Depth (µm)', pad=0.01, aspect=30)
    fig.suptitle(suptitle, y=1.04, fontsize=16)
    plt.show()


plot_subsets_vs_depth("r2_mean", "r2_std",
                      "Ridge Decoder Performance (Mean $R^2$) — cut treadmill",
                      "Mean $R^2$", ylim=(0, 0.9))

In [ ]:
if "mae_mean" in subsets_all_df.columns:
    plot_subsets_vs_depth("mae_mean", "mae_std",
                          "Ridge Decoder Mean Absolute Error (MAE) — cut treadmill",
                          "Mean MAE")

### Layer 2/3 vs Layer 5 (averaged across sessions)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 7))
cond = "closedloop"
max_sizes = subsets_all_df.groupby("session")["subset_size"].transform("max")
filtered = subsets_all_df[subsets_all_df["subset_size"] < max_sizes]

for target in TARGETS:
    cond_df = filtered[(filtered["target"] == target) & (filtered["condition"] == cond)]
    color = TARGET_COLOR[target]; label = TARGET_LABEL[target]
    l23 = cond_df[cond_df.nominal_depth <= CUT_OFF]
    if not l23.empty:
        avg = l23.groupby('subset_size')['r2_mean'].mean()
        ax.plot(avg.index, avg.values, label=f"{label} (L2/3)", marker='o', color=color, linewidth=2)
    l5 = cond_df[cond_df.nominal_depth > CUT_OFF]
    if not l5.empty:
        avg = l5.groupby('subset_size')['r2_mean'].mean()
        ax.plot(avg.index, avg.values, label=f"{label} (L5)", marker='s', color=color, linewidth=2, linestyle='--')

ax.set_title(rf"Decoder Performance by Layer (cut-off: {CUT_OFF} µm)")
ax.set_xlabel("Subset Size"); ax.set_ylabel("Mean $R^2$"); ax.set_xscale("log")
ax.legend(frameon=False, bbox_to_anchor=(1.05, 1), loc='upper left')
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout(); plt.show()

### Cross-target comparison at a fixed subset size

Per-session metrics at subset size 200, compared between targets. The depth vs RS×OF panel
shows the two orthogonal axes side by side.

In [ ]:
subset200 = subsets_all_df[subsets_all_df.subset_size == 200]
pairs = [("OF_stim", "RS_stim"), ("depth", "rsof_product_stim"), ("depth", "OF_stim")]

for metric, mlabel in [("r2_mean", "$r^2$ mean"), ("mae_mean", "MAE")]:
    if metric not in subset200.columns:
        continue
    table = subset200.pivot_table(index=['session', 'nominal_depth'], columns='target', values=metric).reset_index()
    fig, axes = plt.subplots(1, len(pairs), figsize=(5 * len(pairs), 5))
    for ax, (x_col, y_col) in zip(axes, pairs):
        if x_col not in table.columns or y_col not in table.columns:
            ax.set_visible(False); continue
        sc = ax.scatter(table[x_col], table[y_col], c=table['nominal_depth'],
                        cmap='viridis', s=60, edgecolors='none', alpha=0.8)
        ax.set_xlabel(f"{mlabel} {TARGET_LABEL[x_col]}")
        ax.set_ylabel(f"{mlabel} {TARGET_LABEL[y_col]}")
        ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
    fig.suptitle(f"Per-session {mlabel} at subset size 200", y=1.03, fontsize=15)
    plt.tight_layout()
    fig.colorbar(sc, ax=axes.ravel().tolist(), pad=0.02, aspect=20).set_label("Nominal Depth (µm)")
    plt.show()

## 2. Grid 1 vs Grid 2 subset performance

**Grid 1** — balanced high-OF conditions (OF ∈ {64, 256, 1024}).
**Grid 2** — balanced low-OF conditions (OF ∈ {1, 4, 16, 64}).

In [ ]:
GRIDS = ["grid1", "grid2"]
all_data = {}
for grid_name in GRIDS:
    filename = f"ridge_decoder_neuron_subsets_motor_cut_{grid_name}.parquet"
    dfs = []
    for project in PROJECTS:
        df = load_project_subsets(project, filename=filename,
                                  session_to_exclude=SESSIONS_TO_EXCLUDE.keys(), cut_treadmill=True)
        df["project"] = project
        dfs.append(df)
    all_data[grid_name] = pd.concat(dfs, ignore_index=True)
    print(f"{grid_name}: {len(all_data[grid_name])} rows, {all_data[grid_name]['session'].nunique()} sessions")

print("\nTargets:", all_data['grid1']['target'].unique())

In [ ]:
def depth_colormap(depths):
    depths_sorted = np.sort(np.unique(depths))
    norm = mcolors.Normalize(vmin=depths_sorted.min(), vmax=depths_sorted.max())
    return {d: cm.viridis(norm(d)) for d in depths_sorted}


def plot_grid_subsets(ax, df, target, condition="closedloop", title=""):
    """R² vs subset size, one line per session, coloured by nominal depth."""
    sub = df[(df["target"] == target) & (df["condition"] == condition)]
    if sub.empty:
        ax.set_title(title + " (no data)"); return
    color_map = depth_colormap(sub["nominal_depth"].dropna().values)
    for _, sdf in sub.groupby("session"):
        depth = sdf["nominal_depth"].iloc[0]
        sdf = sdf.sort_values("subset_size")
        ax.plot(sdf["subset_size"], sdf["r2_mean"], color=color_map.get(depth, "grey"), alpha=0.7, linewidth=1.5)
    ax.set_xlabel("Subset size (# neurons)"); ax.set_ylabel("Mean R²")
    ax.set_title(title); ax.set_ylim(bottom=0)
    depths_all = sub["nominal_depth"].dropna().unique()
    norm = mcolors.Normalize(vmin=np.min(depths_all), vmax=np.max(depths_all))
    sm = cm.ScalarMappable(cmap=cm.viridis, norm=norm); sm.set_array([])
    plt.colorbar(sm, ax=ax, label="Nominal depth (µm)")

In [ ]:
# One figure per target: Grid 1 vs Grid 2
for target in TARGETS:
    fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=True)
    for ax, grid_name in zip(axes, GRIDS):
        df = all_data[grid_name]
        n_sess = df['session'].nunique()
        plot_grid_subsets(ax, df, target=target, condition="closedloop",
                          title=f"{grid_name} — {TARGET_LABEL[target]} ({n_sess} sessions)")
    fig.suptitle(f"Decode {TARGET_LABEL[target]} from neuronal subsets (closedloop)", fontsize=14)
    fig.tight_layout()
    plt.show()

In [ ]:
# Numeric summary: mean R² at largest subset size per session
rows = []
for grid_name, df in all_data.items():
    for target in TARGETS:
        sub = df[(df["target"] == target) & (df["condition"] == "closedloop")]
        if sub.empty:
            continue
        max_r2 = sub.groupby("session")["r2_mean"].max()
        rows.append({"grid": grid_name, "target": target, "mean_max_r2": max_r2.mean(),
                     "sem_max_r2": max_r2.sem(), "n_sessions": len(max_r2)})
summary_df = pd.DataFrame(rows)
print(summary_df.to_string(index=False))

In [ ]:
# By-layer comparison across both grids
cond = "closedloop"
for grid_name, df in all_data.items():
    max_sizes = df.groupby("session")["subset_size"].transform("max")
    filtered = df[df["subset_size"] < max_sizes]
    fig, ax = plt.subplots(figsize=(10, 7))
    for target in TARGETS:
        cond_df = filtered[(filtered["target"] == target) & (filtered["condition"] == cond)]
        color = TARGET_COLOR[target]; label = TARGET_LABEL[target]
        l23 = cond_df[cond_df.nominal_depth <= CUT_OFF]
        if not l23.empty:
            avg = l23.groupby('subset_size')['r2_mean'].mean()
            ax.plot(avg.index, avg.values, label=f"{label} (L2/3)", marker='o', color=color, linewidth=2)
        l5 = cond_df[cond_df.nominal_depth > CUT_OFF]
        if not l5.empty:
            avg = l5.groupby('subset_size')['r2_mean'].mean()
            ax.plot(avg.index, avg.values, label=f"{label} (L5)", marker='s', color=color, linewidth=2, linestyle='--')
    ax.set_title(rf"Decoder performance for {grid_name} (Layer cut-off: {CUT_OFF} µm)")
    ax.set_xlabel("Subset Size"); ax.set_ylabel("Mean $R^2$"); ax.set_xscale("log")
    ax.legend(frameon=False, bbox_to_anchor=(1.05, 1), loc='upper left')
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
    ax.set_xlim(1, 1000); ax.set_ylim(0, 0.8)
    plt.tight_layout(); plt.show()

### Grid layout reference

In [ ]:
grids_values = dict(
    grid1=[(60.0, 1024.0), (7.0, 256.0), (3.8125, 1024.0), (15.0, 64.0), (3.8125, 64.0),
           (60.0, 256.0), (30.0, 256.0), (3.8125, 256.0), (30.0, 64.0), (15.0, 256.0),
           (7.0, 1024.0), (30.0, 1024.0)],
    grid2=[(60.0, 1.0), (3.8125, 4.0), (60.0, 4.0), (30.0, 4.0), (3.8125, 16.0), (7.0, 16.0),
           (15.0, 16.0), (30.0, 16.0), (60.0, 16.0), (3.8125, 64.0), (7.0, 64.0), (60.0, 64.0)],
)

col = dict(grid1='dodgerblue', grid2='tomato')
fig = plt.figure(figsize=(5, 5))
ax = fig.add_subplot(1, 1, 1, aspect='equal')
for grid_name, conditions in grids_values.items():
    conds = np.vstack(conditions)
    ax.scatter(np.log(conds[:, 0]), np.log(conds[:, 1]), color=col[grid_name], label=grid_name)
allcond = np.vstack(list(grids_values.values()))
rs = np.unique(allcond[:, 0]); ax.set_xticks(np.log(rs)); ax.set_xticklabels(rs)
of = np.unique(allcond[:, 1]); ax.set_yticks(np.log(of)); ax.set_yticklabels(of)
ax.set_xlabel("Running speed (cm/s)"); ax.set_ylabel("Optic flow (deg/s)")
ax.legend(loc='lower right')
plt.tight_layout(); plt.show()